# Advanced Problems with Solutions: Python Imports and `importlib`

This notebook contains advanced exercises on Python imports, `importlib`, module specs, finders, loaders, `sys.path`, dynamic imports, and safe import practices.

Each problem is followed by a complete solution.

## Setup

Run this first. It creates a temporary workspace so the exercises do not modify your real project files.

In [ ]:
import importlib
import importlib.abc
import importlib.machinery
import importlib.util
import pathlib
import shutil
import sys
import tempfile
from contextlib import contextmanager

WORKDIR = pathlib.Path(tempfile.mkdtemp(prefix='importlib_advanced_'))
WORKDIR

## Problem 1 — Dynamic import and namespace binding

Create a module called `plugin_alpha.py` containing:

```python
NAME = 'alpha'

def transform(x):
    return f'alpha:{x}'
```

Then import it using `importlib.import_module('plugin_alpha')`.

Tasks:

1. Add the temporary directory to `sys.path`.
2. Import the module dynamically.
3. Show that the module is stored in `sys.modules`.
4. Show that `importlib.import_module` does not automatically create a local variable named `plugin_alpha`.
5. Fix the problem cleanly.

### Solution 1

In [ ]:
plugin_alpha_path = WORKDIR / 'plugin_alpha.py'
plugin_alpha_path.write_text(
    "NAME = 'alpha'\n\ndef transform(x):\n    return f'alpha:{x}'\n",
    encoding='utf-8'
)

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

globals().pop('plugin_alpha', None)
sys.modules.pop('plugin_alpha', None)

module_obj = importlib.import_module('plugin_alpha')

print('Returned module:', module_obj)
print('Same object in sys.modules:', sys.modules['plugin_alpha'] is module_obj)

try:
    plugin_alpha.transform('data')
except NameError as exc:
    print('Expected NameError:', exc)

plugin_alpha = module_obj
print(plugin_alpha.transform('data'))

## Problem 2 — Inspect module specs

Write a function `describe_module(name)` that returns a dictionary containing:

- `name`
- `found`
- `origin`
- `loader_type`
- `is_package`
- `search_locations`

Use it on:

```python
'sys'
'math'
'fractions'
'plugin_alpha'
'does_not_exist_12345'
```

The function should not raise an exception when the module cannot be found.

### Solution 2

In [ ]:
def describe_module(name):
    spec = importlib.util.find_spec(name)

    if spec is None:
        return {
            'name': name,
            'found': False,
            'origin': None,
            'loader_type': None,
            'is_package': False,
            'search_locations': None
        }

    return {
        'name': name,
        'found': True,
        'origin': spec.origin,
        'loader_type': type(spec.loader).__name__ if spec.loader else None,
        'is_package': spec.submodule_search_locations is not None,
        'search_locations': list(spec.submodule_search_locations) if spec.submodule_search_locations else None
    }

for name in ['sys', 'math', 'fractions', 'plugin_alpha', 'does_not_exist_12345']:
    print(describe_module(name))

## Problem 3 — Diagnose a `sys.path` import failure

Create a directory called `external_plugins`. Inside it, create a module called `plugin_beta.py` containing:

```python
VALUE = 42
```

Tasks:

1. Show that the module cannot be found before the directory is added to `sys.path`.
2. Write `ensure_on_syspath(path)` that adds a directory to `sys.path` exactly once.
3. Show that `find_spec` works after adding the path.
4. Import the module and read `VALUE`.

### Solution 3

In [ ]:
external_plugins = WORKDIR / 'external_plugins'
external_plugins.mkdir(exist_ok=True)

(external_plugins / 'plugin_beta.py').write_text('VALUE = 42\n', encoding='utf-8')

external_str = str(external_plugins)
while external_str in sys.path:
    sys.path.remove(external_str)

sys.modules.pop('plugin_beta', None)

print('Before adding path:', importlib.util.find_spec('plugin_beta'))

def ensure_on_syspath(path):
    path = str(pathlib.Path(path).resolve())
    normalized = [str(pathlib.Path(p).resolve()) for p in sys.path if p]
    if path not in normalized:
        sys.path.insert(0, path)
    return path

ensure_on_syspath(external_plugins)

print('After adding path:', importlib.util.find_spec('plugin_beta'))

plugin_beta = importlib.import_module('plugin_beta')
print(plugin_beta.VALUE)

## Problem 4 — Import a module from an absolute file path

Implement:

```python
def import_from_path(module_name, file_path):
    ...
```

Requirements:

1. Use `importlib.util.spec_from_file_location`.
2. Create the module with `importlib.util.module_from_spec`.
3. Register it in `sys.modules` before execution.
4. Execute the module using the loader.
5. Remove it from `sys.modules` if execution fails.
6. Return the module object.

### Solution 4

In [ ]:
standalone_path = WORKDIR / 'standalone_gamma.py'
standalone_path.write_text(
    "print('executing standalone_gamma')\nVALUE = 'gamma'\n\ndef identity(x):\n    return x\n",
    encoding='utf-8'
)

def import_from_path(module_name, file_path):
    file_path = pathlib.Path(file_path)
    spec = importlib.util.spec_from_file_location(module_name, file_path)

    if spec is None or spec.loader is None:
        raise ImportError(f'Could not create spec for {module_name!r}')

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module

    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(module_name, None)
        raise

    return module

gamma = import_from_path('standalone_gamma', standalone_path)
print(gamma.VALUE)
print(gamma.identity('ok'))
print(sys.modules['standalone_gamma'] is gamma)

## Problem 5 — Safe optional imports

Write a function:

```python
def optional_import(primary, fallback=None):
    ...
```

Behavior:

1. Try to import `primary`.
2. If `primary` is missing and `fallback` exists, import and return the fallback.
3. If `primary` is missing and there is no fallback, return `None`.
4. Do not hide import-time errors raised inside an existing module.

### Solution 5

In [ ]:
broken_path = WORKDIR / 'broken_module.py'
broken_path.write_text(
    "raise RuntimeError('boom during import')\n",
    encoding='utf-8'
)

ensure_on_syspath(WORKDIR)

def optional_import(primary, fallback=None):
    try:
        return importlib.import_module(primary)
    except ModuleNotFoundError as exc:
        if exc.name != primary:
            raise

        if fallback is None:
            return None

        return importlib.import_module(fallback)

print(optional_import('missing_primary_abcxyz', 'math').sqrt(81))
print(optional_import('missing_primary_abcxyz') is None)

try:
    optional_import('broken_module', 'math')
except RuntimeError as exc:
    print('Correctly did not hide import-time error:', exc)

## Problem 6 — Reloading and stale references

Create `reload_target.py` containing:

```python
VERSION = 1

def get_version():
    return VERSION
```

Then:

1. Import it.
2. Save a direct reference to `get_version`.
3. Modify the file so `VERSION = 2`.
4. Call `importlib.invalidate_caches()`.
5. Reload the module.
6. Compare the new module function with the old direct reference.

### Solution 6

In [ ]:
reload_target_path = WORKDIR / 'reload_target.py'
reload_target_path.write_text(
    'VERSION = 1\n\ndef get_version():\n    return VERSION\n',
    encoding='utf-8'
)

ensure_on_syspath(WORKDIR)
sys.modules.pop('reload_target', None)

reload_target = importlib.import_module('reload_target')
old_get_version = reload_target.get_version

print('Before reload:', reload_target.get_version(), old_get_version())

reload_target_path.write_text(
    'VERSION = 2\n\ndef get_version():\n    return VERSION\n',
    encoding='utf-8'
)

importlib.invalidate_caches()
reload_target = importlib.reload(reload_target)

print('After reload, module function:', reload_target.get_version())
print('After reload, old reference:', old_get_version())
print('Same function object:', old_get_version is reload_target.get_version)

## Problem 7 — Build a minimal in-memory importer

Create an importer that allows this to work without creating a physical file:

```python
import virtual_hello
virtual_hello.message()
```

Requirements:

1. Implement a meta path finder.
2. Implement a loader.
3. Return a valid `ModuleSpec`.
4. Execute code into the module namespace.
5. Cleanly install and uninstall the importer.

### Solution 7

In [ ]:
class InMemoryLoader(importlib.abc.Loader):
    def __init__(self, source):
        self.source = source

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        exec(self.source, module.__dict__)


class InMemoryFinder(importlib.abc.MetaPathFinder):
    def __init__(self, modules):
        self.modules = dict(modules)

    def find_spec(self, fullname, path=None, target=None):
        if fullname not in self.modules:
            return None

        loader = InMemoryLoader(self.modules[fullname])
        return importlib.machinery.ModuleSpec(
            name=fullname,
            loader=loader,
            origin='<in-memory>'
        )


@contextmanager
def installed_meta_finder(finder):
    sys.meta_path.insert(0, finder)
    try:
        yield finder
    finally:
        try:
            sys.meta_path.remove(finder)
        except ValueError:
            pass


virtual_modules = {
    'virtual_hello': "VALUE = 'loaded from memory'\n\ndef message():\n    return f'hello: {VALUE}'\n"
}

sys.modules.pop('virtual_hello', None)

finder = InMemoryFinder(virtual_modules)

with installed_meta_finder(finder):
    virtual_hello = importlib.import_module('virtual_hello')
    print(virtual_hello)
    print(virtual_hello.__spec__)
    print(virtual_hello.message())

print('Still cached:', 'virtual_hello' in sys.modules)

sys.modules.pop('virtual_hello', None)

try:
    importlib.import_module('virtual_hello')
except ModuleNotFoundError as exc:
    print('Correctly unavailable after finder removal:', exc)

## Problem 8 — Package imports and `submodule_search_locations`

Create this package:

```text
pkg_demo/
    __init__.py
    tools.py
```

`__init__.py` should contain `PACKAGE_NAME = 'pkg_demo'`.
`tools.py` should contain `tool_name = 'hammer'`.

Tasks:

1. Inspect the spec for `pkg_demo`.
2. Prove it is a package.
3. Import `pkg_demo.tools`.
4. Explain why packages have `submodule_search_locations` but ordinary modules do not.

### Solution 8

In [ ]:
pkg_dir = WORKDIR / 'pkg_demo'
pkg_dir.mkdir(exist_ok=True)

(pkg_dir / '__init__.py').write_text("PACKAGE_NAME = 'pkg_demo'\n", encoding='utf-8')
(pkg_dir / 'tools.py').write_text("tool_name = 'hammer'\n", encoding='utf-8')

ensure_on_syspath(WORKDIR)

sys.modules.pop('pkg_demo', None)
sys.modules.pop('pkg_demo.tools', None)

pkg_spec = importlib.util.find_spec('pkg_demo')
print(pkg_spec)
print('Is package:', pkg_spec.submodule_search_locations is not None)
print('Search locations:', list(pkg_spec.submodule_search_locations))

tools = importlib.import_module('pkg_demo.tools')
print(tools.tool_name)
print('Parent package:', tools.__package__)

## Problem 9 — Detect shadowing of standard library modules

Create a local file named `json.py` in the workspace. Then write:

```python
def detect_shadowing(module_name, suspicious_root):
    ...
```

The function should report whether the module appears to come from the suspicious directory rather than from the expected environment location.

This demonstrates why naming local files after standard library modules is dangerous.

### Solution 9

In [ ]:
shadow_path = WORKDIR / 'json.py'
shadow_path.write_text("SHADOWED = True\n", encoding='utf-8')

ensure_on_syspath(WORKDIR)
sys.modules.pop('json', None)

def detect_shadowing(module_name, suspicious_root):
    spec = importlib.util.find_spec(module_name)
    if spec is None or spec.origin is None:
        return {'module': module_name, 'found': False, 'shadowed': False, 'origin': None}

    origin = pathlib.Path(spec.origin).resolve()
    root = pathlib.Path(suspicious_root).resolve()

    try:
        shadowed = origin.is_relative_to(root)
    except AttributeError:
        shadowed = str(origin).startswith(str(root))

    return {
        'module': module_name,
        'found': True,
        'shadowed': shadowed,
        'origin': str(origin)
    }

print(detect_shadowing('json', WORKDIR))

json = importlib.import_module('json')
print('Imported json object:', json)
print('Is this the fake json?', getattr(json, 'SHADOWED', False))

## Problem 10 — Robust plugin loader

Write a function `load_plugins(directory)` that loads every Python file whose name starts with `plugin_`.

Rules:

1. Ignore files that do not start with `plugin_`.
2. Each plugin must define a callable `register()` function.
3. Return a dictionary with two keys: `loaded` and `errors`.
4. Do not crash when one plugin is broken.
5. Remove failed imports from `sys.modules`.
6. Use unique internal module names to avoid collisions.

### Solution 10

In [ ]:
plugin_dir = WORKDIR / 'real_plugins'
plugin_dir.mkdir(exist_ok=True)

(plugin_dir / 'plugin_good.py').write_text(
    "def register():\n    return {'name': 'good', 'status': 'ok'}\n",
    encoding='utf-8'
)

(plugin_dir / 'plugin_missing_register.py').write_text(
    "VALUE = 'forgot register'\n",
    encoding='utf-8'
)

(plugin_dir / 'plugin_broken.py').write_text(
    "def register():\n    raise ValueError('bad plugin configuration')\n",
    encoding='utf-8'
)

(plugin_dir / 'not_a_plugin.py').write_text(
    "def register():\n    return 'should not load'\n",
    encoding='utf-8'
)

def load_plugins(directory):
    directory = pathlib.Path(directory)
    loaded = {}
    errors = {}

    for file_path in sorted(directory.glob('*.py')):
        stem = file_path.stem

        if not stem.startswith('plugin_'):
            continue

        if not stem.isidentifier():
            errors[stem] = 'Invalid module name'
            continue

        module_name = f'_dynamic_plugins_{directory.name}_{stem}'

        try:
            module = import_from_path(module_name, file_path)
            register = getattr(module, 'register', None)

            if not callable(register):
                raise TypeError('Plugin must define callable register()')

            loaded[stem] = register()

        except Exception as exc:
            sys.modules.pop(module_name, None)
            errors[stem] = f'{type(exc).__name__}: {exc}'

    return {'loaded': loaded, 'errors': errors}

load_plugins(plugin_dir)

## Problem 11 — Compare `__import__` and `importlib.import_module`

Create a package `comparison_pkg` with a submodule `sub.py`.

Compare:

```python
__import__('comparison_pkg.sub')
importlib.import_module('comparison_pkg.sub')
```

What does each call return? Why is `importlib.import_module` usually clearer for dynamic imports?

### Solution 11

In [ ]:
comparison_pkg = WORKDIR / 'comparison_pkg'
comparison_pkg.mkdir(exist_ok=True)

(comparison_pkg / '__init__.py').write_text("PACKAGE_VALUE = 'package'\n", encoding='utf-8')
(comparison_pkg / 'sub.py').write_text("SUB_VALUE = 'submodule'\n", encoding='utf-8')

ensure_on_syspath(WORKDIR)

for name in ['comparison_pkg', 'comparison_pkg.sub']:
    sys.modules.pop(name, None)

via_dunder_import = __import__('comparison_pkg.sub')
via_importlib = importlib.import_module('comparison_pkg.sub')

print('__import__ returned:', via_dunder_import)
print('Has PACKAGE_VALUE:', via_dunder_import.PACKAGE_VALUE)
print('Has SUB_VALUE directly:', hasattr(via_dunder_import, 'SUB_VALUE'))

print('importlib.import_module returned:', via_importlib)
print('SUB_VALUE:', via_importlib.SUB_VALUE)

## Problem 12 — Best-practice review

For each snippet, identify the problem and rewrite it safely.

### Snippet A

```python
try:
    import optional_backend
except Exception:
    pass
```

### Snippet B

```python
sys.path.append('/my/hard-coded/dev/path')
import my_plugin
```

### Snippet C

```python
module = importlib.import_module(user_input)
module.run()
```

### Solution 12

### Snippet A

Problem: it hides every exception, including syntax errors and import-time runtime errors.

Safer version:

```python
try:
    optional_backend = importlib.import_module('optional_backend')
except ModuleNotFoundError as exc:
    if exc.name != 'optional_backend':
        raise
    optional_backend = None
```

### Snippet B

Problem: it mutates global import behavior using a hard-coded path.

Safer version:

```python
plugin_path = pathlib.Path(os.environ['MY_PLUGIN_PATH']).resolve()

if not plugin_path.exists():
    raise FileNotFoundError(plugin_path)

if str(plugin_path) not in sys.path:
    sys.path.insert(0, str(plugin_path))

my_plugin = importlib.import_module('my_plugin')
```

Best practice: package and install plugins properly instead of editing `sys.path` in application code.

### Snippet C

Problem: importing arbitrary user input can execute arbitrary Python code.

Safer version:

```python
ALLOWED_PLUGINS = {
    'csv': 'csv',
    'json': 'json'
}

requested = user_input.strip()

try:
    module_name = ALLOWED_PLUGINS[requested]
except KeyError:
    raise ValueError(f'Unsupported plugin: {requested!r}')

module = importlib.import_module(module_name)
```

Best practice: dynamic import names should come from trusted configuration or a strict allowlist.

## Cleanup

Run this cell when you are finished.

In [ ]:
for name in list(sys.modules):
    if (
        name.startswith('plugin_')
        or name.startswith('standalone_gamma')
        or name.startswith('reload_target')
        or name.startswith('virtual_hello')
        or name.startswith('pkg_demo')
        or name.startswith('_dynamic_plugins_')
        or name.startswith('comparison_pkg')
        or name == 'broken_module'
    ):
        sys.modules.pop(name, None)

for p in [str(WORKDIR), str(WORKDIR / 'external_plugins')]:
    while p in sys.path:
        sys.path.remove(p)

shutil.rmtree(WORKDIR, ignore_errors=True)
print('Cleaned up:', WORKDIR)